# Isolation Forest Anomaly Detection

Isolation Forest is a popular unsupervised anomaly detection model. This notebook creates a small synthetic dataset with outliers, fits the model, and checks how many anomalies it flags.

## Imports and data

In [ ]:
import pandas as pd
from sklearn.datasets import make_blobs
from sklearn.ensemble import IsolationForest

normal_points, _ = make_blobs(
    n_samples=300,
    centers=[(-2, -2), (2, 2)],
    cluster_std=0.6,
    random_state=42,
)
outliers = [[-5, 5], [5, -5], [6, 6], [-6, -6], [0, 6], [6, 0]]

features = pd.DataFrame(
    [*normal_points.tolist(), *outliers],
    columns=["feature_1", "feature_2"],
)
features["is_injected_outlier"] = [False] * len(normal_points) + [True] * len(outliers)

features.head()

## Fit Isolation Forest

In [ ]:
model = IsolationForest(contamination=0.03, random_state=42)
predictions = model.fit_predict(features[["feature_1", "feature_2"]])
scores = model.decision_function(features[["feature_1", "feature_2"]])

results = features.assign(
    anomaly_label=predictions,
    anomaly_score=scores,
    flagged_anomaly=predictions == -1,
)

summary = {
    "flagged_anomalies": int(results["flagged_anomaly"].sum()),
    "injected_outliers": int(results["is_injected_outlier"].sum()),
    "injected_outliers_flagged": int(
        (results["flagged_anomaly"] & results["is_injected_outlier"]).sum()
    ),
}

summary

## Most anomalous points

In [ ]:
results.sort_values("anomaly_score").head(10)

## Practical note

Isolation Forest is useful when labels are unavailable and the goal is to surface unusual records for review. The `contamination` setting controls how aggressively it flags anomalies.